# Mount and move to drive

In [1]:
#mount to drive
from google.colab import drive
!drive.mount('/content/drive')

/bin/bash: -c: line 1: syntax error near unexpected token `'/content/drive''
/bin/bash: -c: line 1: `drive.mount('/content/drive')'


In [2]:
#move to drive location with stored data
%cd /content/drive/MyDrive/CodeData/
%ls

/content/drive/MyDrive/CodeData
snippets-dev/


# Download and unzip dataset

In [3]:
!kaggle datasets download -d simiotic/github-code-snippets-development-sample
!unzip github-code-snippets-development-sample.zip
!rm github-code-snippets-development-sample.zip
%ls

Dataset URL: https://www.kaggle.com/datasets/simiotic/github-code-snippets-development-sample
License(s): Attribution 4.0 International (CC BY 4.0)
User cancelled operation
unzip:  cannot find or open github-code-snippets-development-sample.zip, github-code-snippets-development-sample.zip.zip or github-code-snippets-development-sample.zip.ZIP.
rm: cannot remove 'github-code-snippets-development-sample.zip': No such file or directory
snippets-dev/


#Format DataSet

In [4]:
# Install dependencies as needed:
# pip install kagglehub[hf-datasets]
import kagglehub
from kagglehub import KaggleDatasetAdapter

# Set the path to the file you'd like to load
file_path = "snippets-dev/snippets-dev.db"

# Load the latest version
hf_dataset = kagglehub.dataset_load(
  KaggleDatasetAdapter.HUGGING_FACE,
  "simiotic/github-code-snippets-development-sample",
  file_path, sql_query="SELECT snippet, language FROM snippets"
)

print("Hugging Face Dataset:", hf_dataset)

# Keep only snippets belonging to the languages in filtered_languages
filtered_languages = {'C', 'C++', 'Python', 'Java', 'HTML', 'CSV'}
hf_dataset = hf_dataset.filter(lambda example: example['language'] in filtered_languages)
print("Filtered Dataset:", hf_dataset)

Using Colab cache for faster access to the 'github-code-snippets-development-sample' dataset.
Hugging Face Dataset: Dataset({
    features: ['snippet', 'language'],
    num_rows: 4850000
})
Filtering dataset...


Filter:   0%|          | 0/4850000 [00:00<?, ? examples/s]

In [5]:

# 1. Split 80% for training, 10% for test + validation
train_test_valid = hf_dataset.train_test_split(test_size=0.1, seed=42)

# 2. Split the 10% remaining into 75% test (15% of total) and 25% validation (5% of total)
test_valid = train_test_valid['test'].train_test_split(test_size=0.25, seed=42)

# Gather into a single DatasetDict structure
from datasets import DatasetDict

split_datasets = DatasetDict({
    'train': train_test_valid['train'],
    'test': test_valid['train'],
    'validation': test_valid['test']
})

print(split_datasets)

DatasetDict({
    train: Dataset({
        features: ['snippet', 'language'],
        num_rows: 1462118
    })
    test: Dataset({
        features: ['snippet', 'language'],
        num_rows: 121843
    })
    validation: Dataset({
        features: ['snippet', 'language'],
        num_rows: 40615
    })
})


# Tokenize dataset functions

In [6]:
import tiktoken
num_proc = 4

#grab and define gpt2 encoder
enc = tiktoken.get_encoding("gpt2")
# Define the snippet processing function
def processSnippet(example):
  # Tokenize the language name using the gpt2 encoder
  snippets = enc.encode(example['snippet'])
  snippets.append(enc.eot_token) #append end of text token

  out = {'snippets': snippets, 'len': len(snippets)}
  return out

  # Define the language processing function
def processLanguage(example):
    # Tokenize the language name using the gpt2 encoder
    languages = enc.encode(example['language'])
    languages.append(enc.eot_token)

    out = {'languages': languages, 'len': len(languages)}
    return out

# Tokenize splits

## Tokenize, flatten, and store training data

In [8]:
# Map process functions over dataset to tokenize snippet feature followed by language
tokenized_train = split_datasets['train'].map(processSnippet,
                                      remove_columns=['snippet'],
                                      desc="tokenize code portion of training data")
fullytokenized = tokenized_train.map(
    processLanguage,
    remove_columns=['language'],
    desc="tokenize language portion of training data"
)

#clear collab ram
import gc
del tokenized_train
gc.collect()



tokenize code portion of training data:   0%|          | 0/1462118 [00:00<?, ? examples/s]

tokenize language portion of training data:   0%|          | 0/1462118 [00:00<?, ? examples/s]

In [11]:
#gemini code with slight modifications
import numpy as np

# Flatten the tokenized snippets column into a single flat list
# Using np.uint16 since GPT-2 vocab size is 50,257 (fits within 65,535)
print("Flattening tokenized snippets...")
flat_tokens = [tok for sublist in fullytokenized['snippets'] for tok in sublist]

# Convert to numpy array and save to train.bin
print("Saving to train.bin in the current directory...")
tokens_arr = np.array(flat_tokens, dtype=np.uint16)
tokens_arr.tofile('train.bin')

print("Successfully saved! File size:", len(tokens_arr), "tokens.")

#clear collab
del tokens_arr
del flat_tokens
del fullytokenized
gc.collect()

Flattening tokenized snippets...
Saving to train.bin in the current directory...
Successfully saved! File size: 115020067 tokens.


## Tokenize, flatten, and store test data

In [13]:
# Map process functions over dataset to tokenize snippet feature followed by language
tokenized_test = split_datasets['test'].map(processSnippet,
                                      remove_columns=['snippet'],
                                      desc="tokenize code portion of training data")
fullytokenized = tokenized_test.map(
    processLanguage,
    remove_columns=['language'],
    desc="tokenize language portion of training data"
)

# Flatten the tokenized snippets column into a single flat list
# Using np.uint16 since GPT-2 vocab size is 50,257 (fits within 65,535)
print("Flattening tokenized snippets...")
flat_tokens = [tok for sublist in fullytokenized['snippets'] for tok in sublist]

# Convert to numpy array and save to train.bin
print("Saving to train.bin in the current directory...")
tokens_arr = np.array(flat_tokens, dtype=np.uint16)
tokens_arr.tofile('test.bin')

print("Successfully saved! File size:", len(tokens_arr), "tokens.")

del tokens_arr
del flat_tokens
del fullytokenized
gc.collect()

tokenize code portion of training data:   0%|          | 0/121843 [00:00<?, ? examples/s]

tokenize language portion of training data:   0%|          | 0/121843 [00:00<?, ? examples/s]

Flattening tokenized snippets...
Saving to train.bin in the current directory...
Successfully saved! File size: 9547227 tokens.


## Tokenize, flatten, and store validation data

In [16]:
# Map process functions over dataset to tokenize snippet feature followed by language
tokenized_val = split_datasets['validation'].map(processSnippet,
                                      remove_columns=['snippet'],
                                      desc="tokenize code portion of training data")
fullytokenized = tokenized_val.map(
    processLanguage,
    remove_columns=['language'],
    desc="tokenize language portion of training data"
)

# Flatten the tokenized snippets column into a single flat list
# Using np.uint16 since GPT-2 vocab size is 50,257 (fits within 65,535)
print("Flattening tokenized snippets...")
flat_tokens = [tok for sublist in fullytokenized['snippets'] for tok in sublist]

# Convert to numpy array and save to train.bin
print("Saving to train.bin in the current directory...")
tokens_arr = np.array(flat_tokens, dtype=np.uint16)
tokens_arr.tofile('val.bin')

print("Successfully saved! File size:", len(tokens_arr), "tokens.")
del tokens_arr
del flat_tokens
del fullytokenized
gc.collect()

tokenize code portion of training data:   0%|          | 0/40615 [00:00<?, ? examples/s]

tokenize language portion of training data:   0%|          | 0/40615 [00:00<?, ? examples/s]

Flattening tokenized snippets...
Saving to train.bin in the current directory...
Successfully saved! File size: 3181639 tokens.


88